# Grad-CAM Explainability Analysis (E4, EfficientNet-B0)

Edit the paths in section 0, then run top to bottom in Colab.

## 0. Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/COMP9517/shared_dataset_seed9517")
RESULTS_DIR = Path("/content/drive/MyDrive/COMP9517/deep_learning_results")
CHECKPOINT_PATH = RESULTS_DIR / "E4_efficientnet_b0_pretrained_strong_best.pt"

SPLIT_DIR = DATA_ROOT / "splits"
TEST_CSV = SPLIT_DIR / "test.csv"
IDX_TO_CLASS_PATH = SPLIT_DIR / "idx_to_class.json"
PREDICTIONS_CSV = RESULTS_DIR / "E4_efficientnet_b0_pretrained_strong_predictions.csv"

GRADCAM_DIR = RESULTS_DIR / "analysis_figures" / "gradcam_v2"
TABLE_DIR = RESULTS_DIR / "analysis_tables"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

for p in [DATA_ROOT, TEST_CSV, IDX_TO_CLASS_PATH, PREDICTIONS_CSV, CHECKPOINT_PATH]:
    print(p.exists(), p)


In [ ]:
!pip -q install grad-cam


## 1. Load data and check alignment

test.csv and predictions.csv must be in the same row order. This is checked before anything else runs.

In [ ]:
import json
import numpy as np
import pandas as pd

test_df = pd.read_csv(TEST_CSV)

with open(IDX_TO_CLASS_PATH, "r", encoding="utf-8") as f:
    idx_to_class = json.load(f)

pred_df = pd.read_csv(PREDICTIONS_CSV)

assert len(test_df) == len(pred_df)
assert (test_df["class_index"].to_numpy() == pred_df["target"].to_numpy()).all(), \
    "test.csv and predictions.csv are not row-aligned"

merged_df = test_df.copy()
merged_df["target"] = pred_df["target"].to_numpy()
merged_df["prediction"] = pred_df["prediction"].to_numpy()
merged_df["is_correct"] = merged_df["target"] == merged_df["prediction"]
merged_df["image_full_path"] = merged_df["image_path"].apply(lambda p: str(DATA_ROOT / p))

merged_df.head()


## 2. Load model and transform

Transform must match test_dataloader.py exactly: Resize((224, 224)), no crop.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(idx_to_class)
IMAGE_SIZE = 224

model_gradcam = models.efficientnet_b0(weights=None)
model_gradcam.classifier[1] = nn.Linear(model_gradcam.classifier[1].in_features, NUM_CLASSES)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model_gradcam.load_state_dict(state_dict)
model_gradcam = model_gradcam.to(DEVICE).eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

gradcam_eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

gradcam_display_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
])

target_layers = [model_gradcam.features[-1]]
cam = GradCAM(model=model_gradcam, target_layers=target_layers)


## 3. Helper functions

In [ ]:
def species_label(class_id):
    info = idx_to_class.get(str(int(class_id)), {})
    return info.get("scientific_name", f"class {class_id}"), info.get("common_name", "")


def safe_filename(text):
    return "".join(ch if (ch.isalnum() or ch in "-_") else "_" for ch in str(text))


def load_cam_and_image(image_full_path, class_for_cam):
    image = Image.open(image_full_path).convert("RGB")
    input_tensor = gradcam_eval_transform(image).unsqueeze(0).to(DEVICE)
    display_image = gradcam_display_transform(image)
    rgb_image = np.asarray(display_image).astype(np.float32) / 255.0

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[ClassifierOutputTarget(int(class_for_cam))],
    )[0]

    return grayscale_cam, rgb_image, input_tensor


# fraction of CAM energy inside a centred box, proxy for attention on subject
def cam_center_energy_ratio(cam_map, center_frac=0.6):
    h, w = cam_map.shape
    ch, cw = int(h * center_frac), int(w * center_frac)
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    center_energy = cam_map[y0:y0 + ch, x0:x0 + cw].sum()
    total_energy = cam_map.sum() + 1e-8
    return float(center_energy / total_energy)


# 1 - normalised entropy, higher means attention is more concentrated
def cam_focus_score(cam_map):
    p = cam_map.flatten().astype(np.float64)
    p = p / (p.sum() + 1e-8)
    p = p[p > 0]
    entropy = -(p * np.log(p)).sum()
    max_entropy = np.log(cam_map.size)
    return float(1.0 - entropy / max_entropy)


# cosine similarity between two CAM maps
def cam_overlap(cam_a, cam_b):
    a = cam_a.flatten().astype(np.float64)
    b = cam_b.flatten().astype(np.float64)
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))


## 4. Correct and failed example figures

In [ ]:
def generate_gradcam_figure(row, save_prefix, show=True):
    grayscale_cam, rgb_image, input_tensor = load_cam_and_image(
        row["image_full_path"], row["prediction"]
    )
    visualization = show_cam_on_image(rgb_image, grayscale_cam, use_rgb=True)

    true_name, _ = species_label(row["target"])
    pred_name, _ = species_label(row["prediction"])

    with torch.no_grad():
        probs = torch.softmax(model_gradcam(input_tensor), dim=1)
        confidence = float(probs[0, int(row["prediction"])].item())

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(rgb_image)
    axs[0].set_title(f"Original\nTrue: {true_name}")
    axs[0].axis("off")

    axs[1].imshow(visualization)
    axs[1].set_title(f"Grad-CAM for prediction\nPred: {pred_name}\nConfidence: {confidence:.3f}")
    axs[1].axis("off")

    plt.tight_layout()
    save_path = GRADCAM_DIR / f"{safe_filename(save_prefix)}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close()

    return {
        "image_path": row["image_path"],
        "target": int(row["target"]),
        "prediction": int(row["prediction"]),
        "confidence": confidence,
        "true_scientific_name": true_name,
        "pred_scientific_name": pred_name,
        "center_energy_ratio": cam_center_energy_ratio(grayscale_cam),
        "focus_score": cam_focus_score(grayscale_cam),
        "saved_path": str(save_path),
    }


correct_examples = merged_df[merged_df["is_correct"]].sample(n=3, random_state=9517)
failed_examples = merged_df[~merged_df["is_correct"]].sample(n=3, random_state=9517)

example_records = []
for i, row in correct_examples.reset_index(drop=True).iterrows():
    example_records.append(generate_gradcam_figure(row, f"correct_{i+1}_class_{row['target']}"))
for i, row in failed_examples.reset_index(drop=True).iterrows():
    example_records.append(
        generate_gradcam_figure(row, f"failed_{i+1}_true_{row['target']}_pred_{row['prediction']}")
    )

pd.DataFrame(example_records)


## 5. Correct vs incorrect, quantitative comparison

In [ ]:
from scipy import stats

N_PER_GROUP = 40

correct_pool = merged_df[merged_df["is_correct"]]
incorrect_pool = merged_df[~merged_df["is_correct"]]

correct_sample = correct_pool.sample(n=min(N_PER_GROUP, len(correct_pool)), random_state=9517)
incorrect_sample = incorrect_pool.sample(n=min(N_PER_GROUP, len(incorrect_pool)), random_state=9517)

metric_rows = []
for group_name, df_group in [("correct", correct_sample), ("incorrect", incorrect_sample)]:
    for _, row in df_group.iterrows():
        grayscale_cam, _, input_tensor = load_cam_and_image(row["image_full_path"], row["prediction"])
        with torch.no_grad():
            confidence = float(
                torch.softmax(model_gradcam(input_tensor), dim=1)[0, int(row["prediction"])].item()
            )
        metric_rows.append({
            "image_path": row["image_path"],
            "target": int(row["target"]),
            "prediction": int(row["prediction"]),
            "group": group_name,
            "confidence": confidence,
            "center_energy_ratio": cam_center_energy_ratio(grayscale_cam),
            "focus_score": cam_focus_score(grayscale_cam),
        })

cam_metrics_df = pd.DataFrame(metric_rows)
cam_metrics_path = TABLE_DIR / "E4_gradcam_quantitative_metrics.csv"
cam_metrics_df.to_csv(cam_metrics_path, index=False)

cam_metrics_df.groupby("group")[["confidence", "center_energy_ratio", "focus_score"]].agg(["mean", "std"])


In [ ]:
correct_center = cam_metrics_df.loc[cam_metrics_df["group"] == "correct", "center_energy_ratio"]
incorrect_center = cam_metrics_df.loc[cam_metrics_df["group"] == "incorrect", "center_energy_ratio"]
u_center, p_center = stats.mannwhitneyu(correct_center, incorrect_center, alternative="two-sided")

correct_focus = cam_metrics_df.loc[cam_metrics_df["group"] == "correct", "focus_score"]
incorrect_focus = cam_metrics_df.loc[cam_metrics_df["group"] == "incorrect", "focus_score"]
u_focus, p_focus = stats.mannwhitneyu(correct_focus, incorrect_focus, alternative="two-sided")

print(f"center_energy_ratio: correct={correct_center.mean():.3f}, incorrect={incorrect_center.mean():.3f}, p={p_center:.4f}")
print(f"focus_score: correct={correct_focus.mean():.3f}, incorrect={incorrect_focus.mean():.3f}, p={p_focus:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, title in zip(
    axes,
    ["center_energy_ratio", "focus_score"],
    ["Center energy ratio", "Focus score"],
):
    data_to_plot = [
        cam_metrics_df.loc[cam_metrics_df["group"] == "correct", col],
        cam_metrics_df.loc[cam_metrics_df["group"] == "incorrect", col],
    ]
    ax.boxplot(data_to_plot, labels=["Correct", "Incorrect"])
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.suptitle(f"E4 Grad-CAM metrics, correct vs incorrect (n={len(correct_sample)} per group)")
plt.tight_layout()
path = GRADCAM_DIR / "E4_gradcam_metrics_correct_vs_incorrect.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()


## 6. Top confused species pairs

In [ ]:
from collections import Counter

wrong_df = merged_df[~merged_df["is_correct"]]
pair_counter = Counter(zip(wrong_df["target"], wrong_df["prediction"]))

TOP_K_PAIRS = 3
top_pairs = pair_counter.most_common(TOP_K_PAIRS)
for (t, p), c in top_pairs:
    t_name, _ = species_label(t)
    p_name, _ = species_label(p)
    print(f"{t} ({t_name}) -> {p} ({p_name}): {c}")


In [ ]:
confused_pair_records = []

for (true_id, pred_id), count in top_pairs:
    true_name, _ = species_label(true_id)
    pred_name, _ = species_label(pred_id)

    pair_examples = merged_df[
        (merged_df["target"] == true_id) & (merged_df["prediction"] == pred_id)
    ]
    n_examples = min(2, len(pair_examples))
    examples = pair_examples.sample(n=n_examples, random_state=9517)

    for i, (_, row) in enumerate(examples.reset_index(drop=True).iterrows()):
        true_cam, rgb_image, _ = load_cam_and_image(row["image_full_path"], true_id)
        pred_cam, _, _ = load_cam_and_image(row["image_full_path"], pred_id)
        overlap = cam_overlap(true_cam, pred_cam)

        true_vis = show_cam_on_image(rgb_image, true_cam, use_rgb=True)
        pred_vis = show_cam_on_image(rgb_image, pred_cam, use_rgb=True)

        fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
        axs[0].imshow(rgb_image); axs[0].set_title(f"Original\nTrue: {true_name}"); axs[0].axis("off")
        axs[1].imshow(true_vis); axs[1].set_title(f"CAM true class\n{true_name}"); axs[1].axis("off")
        axs[2].imshow(pred_vis); axs[2].set_title(f"CAM predicted\n{pred_name}\noverlap={overlap:.2f}"); axs[2].axis("off")
        plt.tight_layout()

        save_path = GRADCAM_DIR / f"pair_{true_id}_to_{pred_id}_ex{i+1}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()

        confused_pair_records.append({
            "true_class_id": true_id, "true_species": true_name,
            "predicted_class_id": pred_id, "predicted_species": pred_name,
            "pair_error_count": count,
            "example_image": row["image_path"],
            "true_vs_pred_cam_overlap": overlap,
            "saved_path": str(save_path),
        })

confused_pair_df = pd.DataFrame(confused_pair_records)
confused_pair_path = TABLE_DIR / "E4_top_confused_pairs_gradcam.csv"
confused_pair_df.to_csv(confused_pair_path, index=False)
confused_pair_df


## 7. Summary numbers

In [ ]:
mean_overlap = confused_pair_df["true_vs_pred_cam_overlap"].mean() if len(confused_pair_df) else float("nan")

summary_text = f'''E4 Grad-CAM summary
n = {len(correct_sample)} correct / {len(incorrect_sample)} incorrect

center_energy_ratio: correct={correct_center.mean():.3f}, incorrect={incorrect_center.mean():.3f}, p={p_center:.4f}
focus_score: correct={correct_focus.mean():.3f}, incorrect={incorrect_focus.mean():.3f}, p={p_focus:.4f}
top-{TOP_K_PAIRS} confused pairs mean true-vs-pred CAM overlap: {mean_overlap:.3f}
'''
print(summary_text)

with open(TABLE_DIR / "E4_gradcam_summary_notes.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)
